In [1]:
!pip install transformers torch pandas tqdm


In [2]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Path folder di Google Drive
folder_path = '/content/drive/MyDrive/SKRIPSI MANTAP/data sosmed/.Process/Clean data'

# Path file
file_path = f'{folder_path}/ig_clean.csv'

# Membaca CSV
df = pd.read_csv(
    file_path,
    sep=',',  # gunakan ',' jika disimpan dengan to_csv() default
    engine='python'
)

# Cek nama kolom
print(df.columns)

# Ambil kolom comment
texts = df["ig_clean"].astype(str).tolist()

print("Jumlah komentar:", len(texts))

Index(['ig_clean'], dtype='object')
Jumlah komentar: 25094


In [5]:
model_name = "indolem/indobertweet-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Gunakan GPU jika tersedia
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()


config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/235k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  445MB            

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: indolem/indobertweet-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B /  445MB            

model.safetensors: downloading bytes:           |  0.00B            

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(31923, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [6]:
def indobertweet_embedding(text_list, max_length=50):
    embeddings = []

    for text in tqdm(text_list):
        encoded = tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        encoded = {k: v.to(device) for k, v in encoded.items()}

        with torch.no_grad():
            output = model(**encoded)

        # Mean Pooling
        token_embeddings = output.last_hidden_state
        attention_mask = encoded["attention_mask"]

        mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        masked_embeddings = token_embeddings * mask

        summed = torch.sum(masked_embeddings, dim=1)
        counts = torch.clamp(mask.sum(dim=1), min=1e-9)

        mean_pooled = summed / counts
        embeddings.append(mean_pooled.squeeze().cpu().numpy())

    return embeddings


In [7]:
embeddings = indobertweet_embedding(texts)

print("Jumlah embedding:", len(embeddings))
print("Dimensi embedding:", embeddings[0].shape)


Streaming output truncated to the last 5000 lines.
 13%|█▎        | 3347/25094 [00:40<03:42, 97.75it/s]

 13%|█▎        | 3358/25094 [00:40<03:40, 98.74it/s]

 13%|█▎        | 3369/25094 [00:40<03:37, 99.92it/s]

 13%|█▎        | 3380/25094 [00:40<03:35, 100.81it/s]

 14%|█▎        | 3391/25094 [00:40<03:35, 100.93it/s]

 14%|█▎        | 3402/25094 [00:40<03:34, 101.15it/s]

 14%|█▎        | 3413/25094 [00:40<03:42, 97.27it/s] 

 14%|█▎        | 3424/25094 [00:40<03:39, 98.90it/s]

 14%|█▎        | 3434/25094 [00:41<03:40, 98.41it/s]

 14%|█▎        | 3445/25094 [00:41<03:36, 99.89it/s]

 14%|█▍        | 3456/25094 [00:41<03:34, 100.69it/s]

 14%|█▍        | 3467/25094 [00:41<03:32, 101.63it/s]

 14%|█▍        | 3478/25094 [00:41<03:35, 100.52it/s]

 14%|█▍        | 3489/25094 [00:41<03:33, 101.05it/s]

 14%|█▍        | 3500/25094 [00:41<03:36, 99.58it/s] 

 14%|█▍        | 3510/25094 [00:41<03:38, 98.68it/s]

 14%|█▍        | 3520/25094 [00:41<03:49, 93.97it/s]

 14%|█▍        | 3530/

Jumlah embedding: 25094
Dimensi embedding: (768,)


In [8]:
embedding_df = pd.DataFrame(embeddings)
embedding_df.columns = [f"emb_{i}" for i in range(embedding_df.shape[1])]

# Gabungkan dengan data asli
final_df = pd.concat([df.reset_index(drop=True), embedding_df], axis=1)

# Simpan hasil embedding
final_df.to_csv("ig_embed_lightgbm.csv", index=False)

print("Embedding berhasil disimpan!")


Embedding berhasil disimpan!


In [9]:
!zip ig_embed_lightgbm.zip ig_embed_lightgbm.csv


  adding: ig_embed_lightgbm.csv (deflated 57%)


In [10]:
from google.colab import files

files.download("ig_embed_lightgbm.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
import numpy as np

np.save("ig_embed_lightgbm.npy", embeddings)
from google.colab import files

files.download("ig_embed_lightgbm.npy")



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
!zip ig_embed_lightgbm.zip ig_embed_lightgbm.npy


  adding: ig_embed_lightgbm.npy (deflated 7%)


In [13]:
import os

folder_path = '/content/drive/MyDrive/SKRIPSI MANTAP/data sosmed/Embedded data'

os.makedirs(folder_path, exist_ok=True)

Simpan ke gdrive

In [14]:
output_path = f'{folder_path}/ig_embed_lightgbm.csv'

final_df.to_csv(output_path, index=False)

print("File berhasil disimpan di:")
print(output_path)

File berhasil disimpan di:
/content/drive/MyDrive/SKRIPSI MANTAP/data sosmed/Embedded data/ig_embed_lightgbm.csv
